# Extract package data from PCAPs

In [1]:
import pathlib
import re

import polars

from utils import list_code

EVALUATION_DIR = pathlib.Path.cwd()
INPUT_PATH = EVALUATION_DIR / "output_dataset"

TBD:
- list code
- **Parallel needs to be installed for this!**
- wireshark preferences:
  ```yaml
  dtls.psk: f28aa24f8cb8a7cc483cf336c0be70047f29
  ```
  wireshark oscore_contexts:
  ```csv
  "01","64617461","f28aa24f8cb8a7cc483cf336c0be70047f29","","","AES-CCM-16-64-128 (CCM*)"
  "64617461","01","f28aa24f8cb8a7cc483cf336c0be70047f29","","","AES-CCM-16-64-128 (CCM*)"
  "02","646e73","0e08f702e6508657e3a0fd996a042531fc89","","","AES-CCM-16-64-128 (CCM*)"
  "646e73","02","0e08f702e6508657e3a0fd996a042531fc89","","","AES-CCM-16-64-128 (CCM*)"
  "03","70726f7879","b35c9719b3594553315a210b18720da0e750","","","AES-CCM-16-64-128 (CCM*)"
  "70726f7879","03","b35c9719b3594553315a210b18720da0e750","","","AES-CCM-16-64-128 (CCM*)"
  ```

In [2]:
list_code(EVALUATION_DIR / "extract_from_pcaps.sh")

#!/usr/bin/env bash
#
# extract_from_pcaps.sh
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR=$( cd -- "$( dirname -- "$(realpath "$0")" )" &> /dev/null && pwd )
PROCS=$(grep -c '^processor' /proc/cpuinfo)
if [ $PROCS -gt 64 ]; then
    # leave some resources to collegues ;-)
    PROCS=$(( (PROCS * 3) / 4))
fi
OUTPUT_DATASET="${OUTPUT_DATASET:-${SCRIPT_DIR}/output_dataset}"


extract_from_pcap() {
    PCAP="$1"

    if ! echo "$PCAP" | grep -q "oscore.*\.upstream.pcapng"; then
        "${SCRIPT_DIR}/extract_eth.sh" "${PCAP}" > "${PCAP%.pcapng}".eth.csv
    fi
    "${SCRIPT_DIR}/extract_coap.sh" "${PCAP}" > "${PCAP%.pcapng}".coap.csv
}

export -f extract_from_pcap
export SCRIPT_DIR

find "${OUTPUT_DATASET}" -name *.wpan.pcapng -o -name oscore*.upstream.pcapng |
    parallel -j"${PROCS}" extract_from_pcap

In [3]:
list_code(EVALUATION_DIR / "extract_eth.sh")

#! /bin/sh
#
# extract_eth.sh
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.
#

if [ $# -ne 1 ]; then
    echo "usage: $0 <pcap file>" >&2
    exit 1
fi

PCAP="$1"

FIELDS="frame.number frame.time_epoch eth.payload"
echo "${FIELDS}" | \
    awk 'BEGIN {OFS="\t"} { for (i = 1; i <= NF; i++) { printf "%s%s", $i, (i < NF) ? OFS : ORS } }'
tshark -Tfields -e frame.number -e frame.time_epoch -e data --disable-protocol ALL --enable-protocol eth -r "${PCAP}"

In [4]:
list_code(EVALUATION_DIR / "extract_coap.sh")

#! /bin/sh
#
# extract_eth.sh
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.
#

if [ $# -ne 1 ]; then
    echo "usage: $0 <pcap file>" >&2
    exit 1
fi

PCAP="$1"

FIELDS="frame.number frame.time_epoch frame.protocols"
FIELDS="${FIELDS} coap.code coap.token coap.opt.ctype coap.opt.block_number coap.opt.block_mflag coap.opt.block_size"

if echo "$PCAP" | grep -q "oscore"; then
    FIELDS="${FIELDS} oscore.code oscore.opt.ctype oscore.opt.block_number oscore.opt.block_mflag oscore.opt.block_size"
fi


echo "${FIELDS}" | \
    awk 'BEGIN {OFS="\t"} { for (i = 1; i <= NF; i++) { printf "%s%s", $i, (i < NF) ? OFS : ORS } }'
tshark -Tfields $(for field in ${FIELDS}; do printf -- "-e %s " "${field}"; done) -r "${PCAP}"

In [5]:
!"{EVALUATION_DIR}"/extract_from_pcaps.sh

In [6]:
COLUMN_ORDER = [
    "client.coap.response.recv_time_epoch",
    "client.type",
    "client.dns.qry.name",
    "client.dns.qry.type",
    "client.coap.url",
    "client.coap.media_type",
    "client.coap.response.code",
    "client.coap.response.payload",
    "frame.number",
    "frame.time_epoch",
    "frame.protocols",
    "eth.payload",
    "ipv6.prefix",
    "coap.is_response",
    "coap.code",
    "coap.token",
    "coap.opt.ctype",
    "coap.opt.block_number",
    "coap.opt.block_mflag",
    "coap.opt.block_size",
    "oscore.code",
    "oscore.opt.ctype",
    "oscore.opt.block_number",
    "oscore.opt.block_mflag",
    "oscore.opt.block_size",
    "upstream.frame.number",
    "upstream.frame.time_epoch",
    "upstream.frame.protocols",
    "upstream.ipv6.prefix",
    "upstream.coap.code",
    "upstream.coap.token",
    "upstream.coap.opt.ctype",
    "upstream.coap.opt.block_number",
    "upstream.coap.opt.block_mflag",
    "upstream.coap.opt.block_size",
    "upstream.oscore.code",
    "upstream.oscore.opt.ctype",
    "upstream.oscore.opt.block_number",
    "upstream.oscore.opt.block_mflag",
    "upstream.oscore.opt.block_size",
]

tables = {}

for root, dirs, files in INPUT_PATH.walk():
    for file in files:
        if not re.search(
            r"json_dns_message",
            file,
        ):
            continue
        if match := re.search(
            r".*\.((wpan|upstream)\.(coap|eth)\.csv|(client|proxy)\.log)",
            file,
        ):
            scenario = file.split(".")[0]
            if scenario not in tables:
                tables[scenario] = {}
            if not file.startswith("oscore") and (
                match[1] == "upstream"
                or match[4] == "proxy"
            ):
                continue
            elif match[1] == "client.log":
                table = "client"
            elif match[1] == "proxy.log":
                table = "proxy"
            else:
                table = match[3]
            if match[2] == "upstream":
                table = f"{table}_upstream"
            tables[scenario][table] = polars.read_csv(
                root / file,
                separator="\t"
            )

from IPython.display import display

for scenario in tables:
    assert all(table in tables[scenario] for table in ["client", "eth", "coap"])
    assert "coap_upstream" not in tables[scenario] or "proxy" in tables[scenario]
    df = tables[scenario]["eth"].join(
        tables[scenario]["coap"][
            [
                col for col in tables[scenario]["coap"].columns
                if col != "frame.time_epoch"
            ]
        ],
        on="frame.number",
        how="inner"
    )
    if "coap_upstream" in tables[scenario]:
        df = df.with_columns(((df["coap.code"] & (7 << 5)) != 0).alias("coap.is_response"))
        upstream_df = tables[scenario]["coap_upstream"]
        upstream_df = upstream_df.with_columns(((upstream_df["coap.code"] & (7 << 5)) != 0).alias("coap.is_response"))
        df = df.join(
            tables[scenario]["proxy"],
            left_on="coap.token",
            right_on="old_token",
            how="left",
        )
        df = df.join(
            upstream_df,
            left_on=["coap.is_response", "new_token"],
            right_on=["coap.is_response", "coap.token"],
            how="left",
            suffix=".upstream",
        )
        df = df.rename({"new_token": "coap.token.upstream"}).join(
            tables[scenario]["client"],
            left_on="coap.token.upstream",
            right_on="token",
        ).rename(
            lambda c: f"upstream.{c[:-9]}" if c.endswith(".upstream") else c
        ).rename(
            {
                "timestamp": "client.coap.response.recv_time_epoch",
                "wpan_prefix": "ipv6.prefix",
                "upstream_prefix": "upstream.ipv6.prefix",
                "type": "client.type",
                "query_name": "client.dns.qry.name",
                "query_type": "client.dns.qry.type",
                "url": "client.coap.url",
                "media_type": "client.coap.media_type",
                "response_code": "client.coap.response.code",
                "response_payload": "client.coap.response.payload",
            }
        )
    else:
        df = df.join(
            tables[scenario]["client"],
            left_on="coap.token",
            right_on="token",
        ).rename(
            {
                "timestamp": "client.coap.response.recv_time_epoch",
                "wpan_prefix": "ipv6.prefix",
                "upstream_prefix": "upstream.ipv6.prefix",
                "type": "client.type",
                "query_name": "client.dns.qry.name",
                "query_type": "client.dns.qry.type",
                "url": "client.coap.url",
                "media_type": "client.coap.media_type",
                "response_code": "client.coap.response.code",
                "response_payload": "client.coap.response.payload",
            }
        )
    assert all(c in COLUMN_ORDER for c in df.columns)
    df = df.select([c for c in COLUMN_ORDER if c in df.columns])
    df.write_csv(INPUT_PATH / f"{scenario}.merged.csv", separator=";")
    max_len = df["eth.payload"].str.len_chars().max()
    df.select(
        ["frame.number", "frame.time_epoch", "client.type", "eth.payload"]
    ).with_columns(
        **{
            "eth.payload": polars.col("eth.payload").str.pad_end(max_len, "x")
        }
    ).write_csv(
        INPUT_PATH / f"{scenario}.training.csv", separator=";"
    )